# AfroViewz 07: Evaluation and error analysis

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JBWandera/Capstone/blob/afroviewz/notebooks/07_evaluation.ipynb)

**Workflow stage:** evaluation. **Runtime:** CPU is enough (it reads saved predictions).

Every run is scored the same way:
1. **mAP on all predictions** down to score 0.001 (the v1 notebook cut at 0.5 first, which understates AP);
2. a **confidence threshold chosen on the validation split** (best F1), then applied unchanged to test for precision, recall and F1;
3. **95% bootstrap intervals** (resampling test images) so small differences are not over-read;
4. results **per source** and **per pothole size at the model's input**;
5. a look at the mistakes.

## Setup

In [ ]:
# Works on Colab and locally. On Colab: mounts Drive, gets the code, installs it.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/Capstone"):
        subprocess.run(["git", "clone", "-q", "-b", "afroviewz", "https://github.com/JBWandera/Capstone.git", "/content/Capstone"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/Capstone", "pull", "-q"], check=True)
    os.chdir("/content/Capstone")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"], check=True)
    sys.path.insert(0, "/content/Capstone/src")
from afroviewz import paths
print("data root:", paths.data_root())


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from afroviewz.viz import show_jpeg
from afroviewz.data import coco as C
from afroviewz.eval import detection as E
from afroviewz.train import runner

DATASET_DIR = Path("/content/datasets/afroviewz-v1/rfdetr") if IN_COLAB else paths.processed_dir("afroviewz-v1") / "rfdetr"
gt = {s: C.load(DATASET_DIR / s / "_annotations.coco.json") for s in ("valid", "test")}
RUNS = runner.runs_root()
runs = sorted(d.name for d in RUNS.iterdir() if (d / "preds_test.json").exists())
print("runs with predictions:", runs)


## 1. Headline table

In [ ]:
def resolution_of(run):
    try:
        cfg = runner.experiment_config(run)
        return cfg.get("resolution") or cfg.get("imgsz") or 560
    except KeyError:
        return 560  # E0, the v1 RF-DETR Base

rows, thresholds = [], {}
for run in runs:
    pv = json.loads((RUNS / run / "preds_valid.json").read_text())
    pt = json.loads((RUNS / run / "preds_test.json").read_text())
    tv = E.threshold_table(gt["valid"], pv)
    thr = E.pick_threshold(tv)
    thresholds[run] = thr
    tt = E.threshold_table(gt["test"], pt, thresholds=np.array([thr])).iloc[0]
    m = E.coco_metrics(gt["test"], pt)
    timing = json.loads((RUNS / run / "timing.json").read_text())
    rows.append({"run": run, "input_px": resolution_of(run), "mAP50:95": m["AP"], "mAP50": m["AP50"],
                 "mAP75": m["AP75"], "threshold (from valid)": thr, "precision": tt.precision,
                 "recall": tt.recall, "F1": tt.f1, "ms/img (T4)": timing.get("test")})
headline = pd.DataFrame(rows).set_index("run").round(3).sort_values("mAP50", ascending=False)
headline.to_csv(RUNS / "headline_test.csv")
headline


## 2. How sure are these numbers? 95% bootstrap intervals

In [ ]:
ci_rows = []
for run in runs:
    pt = json.loads((RUNS / run / "preds_test.json").read_text())
    ci = E.bootstrap(gt["test"], pt, n=100, metrics=("AP", "AP50"))
    ci_rows.append({"run": run, "mAP50": headline.loc[run, "mAP50"],
                    "mAP50 95% CI": f"{ci.loc['AP50', 'low']:.3f} to {ci.loc['AP50', 'high']:.3f}",
                    "mAP50:95": headline.loc[run, "mAP50:95"],
                    "mAP50:95 95% CI": f"{ci.loc['AP', 'low']:.3f} to {ci.loc['AP', 'high']:.3f}"})
pd.DataFrame(ci_rows).set_index("run")


## 3. Results per source

In [ ]:
def by_source(split_coco):
    out = {}
    for im in split_coco["images"]:
        out.setdefault(im["file_name"].split("__")[0], []).append(im["id"])
    return out

src_ids = by_source(gt["test"])
rows = []
for run in runs:
    pt = json.loads((RUNS / run / "preds_test.json").read_text())
    for src, ids in src_ids.items():
        sub = C.subset(gt["test"], ids)
        if not sub["annotations"]:
            continue
        idset = set(ids)
        m = E.coco_metrics(sub, [p for p in pt if p["image_id"] in idset])
        rows.append({"run": run, "source": src, "mAP50": m["AP50"]})
pd.DataFrame(rows).pivot(index="run", columns="source", values="mAP50").round(3)


## 4. Recall by pothole size at the model's input

Recall at the chosen threshold, split by how big each pothole is after the image is resized to the model's input. This is the direct test of whether higher input resolution (E3) helps with small potholes.

In [ ]:
rows = []
for run in runs:
    pt = json.loads((RUNS / run / "preds_test.json").read_text())
    res = resolution_of(run)
    bins = E.size_bins_at(gt["test"], res)
    thr = thresholds[run]
    for size in ("small", "medium", "large"):
        ids = {a for a, b in bins.items() if b == size}
        sub = dict(gt["test"], annotations=[a for a in gt["test"]["annotations"] if a["id"] in ids])
        if not sub["annotations"]:
            continue
        m, n = E.match_predictions(sub, [p for p in pt if p["score"] >= thr])
        rows.append({"run": run, "size": size, "potholes": n, "recall": m.tp.sum() / n})
sizes = pd.DataFrame(rows)
sizes.pivot(index="run", columns="size", values="recall").round(3)[["small", "medium", "large"]]


## 5. Precision and recall against the threshold (validation)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for run in runs:
    pv = json.loads((RUNS / run / "preds_valid.json").read_text())
    t = E.threshold_table(gt["valid"], pv, thresholds=np.round(np.arange(0.02, 0.99, 0.02), 2))
    ax.plot(t.recall, t.precision, label=run)
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.legend(fontsize=8); ax.set_title("Precision and recall, validation split, IoU 0.5")
plt.show()


## 6. Mistakes of the best run

In [ ]:
from PIL import Image
import matplotlib.patches as patches

best = headline.index[0]
pt = json.loads((RUNS / best / "preds_test.json").read_text())
thr = thresholds[best]
keep = [p for p in pt if p["score"] >= thr]
m, _ = E.match_predictions(gt["test"], keep)
fp_imgs = m[~m.tp].image_id.value_counts().index[:4].tolist()
gt_by = {}
for a in gt["test"]["annotations"]:
    gt_by.setdefault(a["image_id"], []).append(a["bbox"])
pred_by = {}
for p in keep:
    pred_by.setdefault(p["image_id"], []).append(p["bbox"])
recall_per_img = {i: m[(m.image_id == i) & m.tp].shape[0] / len(b) for i, b in gt_by.items()}
fn_imgs = sorted(recall_per_img, key=recall_per_img.get)[:4]
ims = {im["id"]: im for im in gt["test"]["images"]}

def panel(ids, title):
    fig, axes = plt.subplots(1, len(ids), figsize=(4.5 * len(ids), 3.6))
    for ax, i in zip(np.atleast_1d(axes), ids):
        ax.imshow(Image.open(DATASET_DIR / "test" / ims[i]["file_name"]))
        for x, y, w, h in gt_by.get(i, []):
            ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, lw=2, ec="#22c55e"))
        for x, y, w, h in pred_by.get(i, []):
            ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, lw=2, ec="#f97316", ls="--"))
        ax.set_axis_off(); ax.set_title(ims[i]["file_name"].split("__")[0], fontsize=9)
    fig.suptitle(f"{title}  (green: label, orange dashed: {best})")
    plt.tight_layout(); show_jpeg(fig)

panel(fp_imgs, "Most false alarms")
panel(fn_imgs, "Most missed potholes")
